# 🎨 VQ-VAE Part A: Turning Images into Tokens

## The Big Idea: Images as Sequences of Tokens

Language models work by turning text into a sequence of discrete tokens from a fixed vocabulary. VQ-VAE does the exact same thing to images.

**The codebook analogy:** Imagine you have a box of 512 emoji stickers 🟥🟦🟩🟨🟧. Any image can be described by picking which stickers to place where — "top-left looks like 🟦, center looks like 🟥". The codebook *is* that sticker box.

**The full pipeline:**
```
Image (3×32×32)
    → Encoder (CNN)         → continuous latent (D×8×8)
    → Quantizer (codebook)  → discrete indices (8×8), each in {0..K-1}
    → Decoder (CNN)         → reconstructed image (3×32×32)
```

**Why this unlocks Transformers for images:**
- An 8×8 grid of indices = a sequence of 64 tokens
- A Transformer (e.g. GPT) can model `p(token_1, token_2, ..., token_64)` autoregressively
- At inference: sample token sequence → decode through codebook + decoder → full image
- This is exactly what **DALL-E v1** and **VQ-GAN** do

In [ ]:
import torch
import torch.nn as nn

class VQVAEEncoder(nn.Module):
    """Compress 3x32x32 image → embedding_dim x 8x8 continuous latents."""
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # → 32x16x16
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # → 64x8x8
            nn.ReLU(),
            nn.Conv2d(64, embedding_dim, 1),             # → embedding_dim x 8x8
        )

    def forward(self, x):
        print(f"  Input:   {tuple(x.shape)}")
        for layer in self.net:
            x = layer(x)
            if isinstance(layer, nn.Conv2d):
                print(f"  After {layer}: {tuple(x.shape)}")
        return x

encoder = VQVAEEncoder(embedding_dim=64)
x = torch.randn(2, 3, 32, 32)   # batch of 2 images
print("=== Encoder shapes ===")
z_e = encoder(x)
print(f"  Final z_e: {tuple(z_e.shape)}  ← continuous latent")

In [ ]:
class VectorQuantizer(nn.Module):
    """
    Map continuous latents to nearest codebook entry.
    Straight-through estimator lets gradients flow back through the argmin.
    """
    def __init__(self, num_embeddings=512, embedding_dim=64, commitment_cost=0.25):
        super().__init__()
        self.K = num_embeddings
        self.D = embedding_dim
        self.beta = commitment_cost
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)  # K × D
        nn.init.uniform_(self.codebook.weight, -1/self.K, 1/self.K)

    def forward(self, z_e):
        B, D, H, W = z_e.shape
        # Flatten spatial dims: (B*H*W, D)
        flat = z_e.permute(0,2,3,1).reshape(-1, self.D)

        # Squared L2 distances to every codebook entry
        dists = (flat.pow(2).sum(1, keepdim=True)
                 - 2 * flat @ self.codebook.weight.T
                 + self.codebook.weight.pow(2).sum(1))

        indices = dists.argmin(dim=1)                         # (B*H*W,)
        z_q = self.codebook(indices).reshape(B, H, W, D).permute(0,3,1,2)  # (B,D,H,W)

        # Losses
        codebook_loss = (z_q.detach() - z_e).pow(2).mean()   # move codebook toward z_e
        commitment_loss = (z_q - z_e.detach()).pow(2).mean()  # move z_e toward codebook

        # Straight-through: copy z_q values but route gradients through z_e
        z_q_st = z_e + (z_q - z_e).detach()

        print(f"  Indices shape: {tuple(indices.reshape(B,H,W).shape)}  values in [0, {self.K-1}]")
        print(f"  Unique codes used: {indices.unique().numel()} / {self.K}")
        return z_q_st, indices.reshape(B, H, W), codebook_loss, commitment_loss

quantizer = VectorQuantizer(num_embeddings=512, embedding_dim=64)
print("=== Quantizer ===")
z_q, indices, cb_loss, commit_loss = quantizer(z_e)
print(f"  z_q shape: {tuple(z_q.shape)}  ← quantized latent (same shape as z_e)")
print(f"  codebook_loss={cb_loss.item():.4f},  commitment_loss={commit_loss.item():.4f}")

In [ ]:
class VQVAEDecoder(nn.Module):
    """Upsample embedding_dim x 8x8 → 3x32x32 reconstructed image."""
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(embedding_dim, 64, 1),                       # → 64x8x8
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # → 32x16x16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # → 3x32x32
            nn.Tanh(),
        )

    def forward(self, z_q):
        print(f"  Input z_q: {tuple(z_q.shape)}")
        x = z_q
        for layer in self.net:
            x = layer(x)
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
                print(f"  After {layer}: {tuple(x.shape)}")
        return x

decoder = VQVAEDecoder(embedding_dim=64)
print("=== Decoder shapes ===")
x_recon = decoder(z_q)
print(f"  Reconstructed: {tuple(x_recon.shape)}  ← matches original input")

## The Four Loss Functions

VQ-VAE training balances four objectives:

### 1. Reconstruction Loss (MSE)
Pixel-level fidelity — did we get the image back?
$$\mathcal{L}_{\text{recon}} = \|x - \hat{x}\|^2$$

### 2. Quantization Losses
Two paired losses keep the encoder and codebook close to each other:
- **Codebook loss** — pulls codebook entries toward encoder outputs (stop gradient on $z_e$):
  $\mathcal{L}_{\text{cb}} = \|\text{sg}[z_e] - z_q\|^2$
- **Commitment loss** — pulls encoder outputs toward codebook (stop gradient on $z_q$), scaled by $\beta$:
  $\mathcal{L}_{\text{commit}} = \beta \|z_e - \text{sg}[z_q]\|^2$

### 3. Perceptual Loss (VGG features)
MSE in VGG feature space — penalizes perceptually visible differences that pixel MSE ignores:
$$\mathcal{L}_{\text{perc}} = \|\phi(x) - \phi(\hat{x})\|^2$$

### 4. Adversarial Loss (GAN discriminator)
A discriminator $D$ is trained alongside the decoder. The decoder tries to fool it:
$$\mathcal{L}_{\text{adv}} = -\mathbb{E}[\log D(\hat{x})]$$

### Weighted Total
$$\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{\text{recon}} + \mathcal{L}_{\text{cb}} + \beta \mathcal{L}_{\text{commit}} + \lambda_2 \mathcal{L}_{\text{perc}} + \lambda_3 \mathcal{L}_{\text{adv}}$$

Typical weights: $\lambda_1=1.0$, $\beta=0.25$, $\lambda_2=0.1$, $\lambda_3=0.1$ — reconstruction dominates early, adversarial takes over later for sharpness.

In [ ]:
import torch.nn.functional as F

# --- Loss weights ---
lambda_recon = 1.0
beta          = 0.25   # commitment cost
lambda_perc  = 0.1
lambda_adv   = 0.1

# --- 1. Reconstruction loss (MSE) ---
l_recon = F.mse_loss(x_recon, x)

# --- 2. Quantization losses (from the quantizer) ---
l_codebook  = cb_loss
l_commit    = beta * commit_loss

# --- 3. Perceptual loss (VGG features) ---
# In practice: vgg = torchvision.models.vgg16(pretrained=True).features[:16]
# Here we simulate with random feature maps of the same spatial resolution
feat_real  = torch.randn(2, 256, 8, 8)   # placeholder for phi(x)
feat_fake  = torch.randn(2, 256, 8, 8)   # placeholder for phi(x_hat)
l_perc = F.mse_loss(feat_fake, feat_real)

# --- 4. Adversarial loss (generator side) ---
# In practice: disc_logits = discriminator(x_recon)
# Here we simulate with random logits
disc_logits = torch.randn(2, 1)           # placeholder
l_adv = -disc_logits.mean()               # generator wants D(x_hat) → 1

# --- Weighted total ---
l_total = (lambda_recon * l_recon
           + l_codebook
           + l_commit
           + lambda_perc * l_perc
           + lambda_adv  * l_adv)

print(f"Reconstruction : {lambda_recon}   × {l_recon.item():.4f}  = {lambda_recon * l_recon.item():.4f}")
print(f"Codebook       :  1.00  × {l_codebook.item():.4f}  = {l_codebook.item():.4f}")
print(f"Commitment     : {beta}   × {commit_loss.item():.4f}  = {l_commit.item():.4f}")
print(f"Perceptual     : {lambda_perc}   × {l_perc.item():.4f}  = {lambda_perc * l_perc.item():.4f}")
print(f"Adversarial    : {lambda_adv}   × {l_adv.item():.4f}  = {lambda_adv * l_adv.item():.4f}")
print(f"{'─'*50}")
print(f"Total loss     :                       {l_total.item():.4f}")

## Quiz

**Q1: Why can't we just use standard backpropagation through the argmin (nearest-neighbor lookup)?**

<details>
<summary>Show answer</summary>

The argmin operation has zero gradient almost everywhere — it's a step function. The straight-through estimator sidesteps this by pretending the gradient of $z_q$ is the same as the gradient of $z_e$ in the backward pass: `z_q_st = z_e + (z_q - z_e).detach()`. The forward pass uses $z_q$ values; the backward pass routes gradients directly through $z_e$.

</details>

---

**Q2: What is codebook collapse, and how can it be detected?**

<details>
<summary>Show answer</summary>

Codebook collapse is when only a small fraction of the $K$ codebook entries are ever used — the rest are dead. It is detected by monitoring **codebook utilization**: the number of unique indices used in a batch divided by $K$. Low utilization (e.g., 50/512 active codes) means the quantizer has collapsed. Fixes include exponential moving average (EMA) codebook updates, random codebook reinitialization of dead entries, and adding noise to encoder outputs.

</details>

---

**Q3: After training a VQ-VAE, how would you generate a completely new image (not reconstruct an existing one)?**

<details>
<summary>Show answer</summary>

Train a separate **prior** — typically a Transformer (like GPT) — to model the distribution of index sequences seen during VQ-VAE training. At inference time: (1) sample an index sequence of length $H \times W$ autoregressively from the Transformer, (2) look up each index in the codebook to get a $D \times H \times W$ latent, (3) pass the latent through the VQ-VAE decoder to produce the final image. This is the DALL-E v1 / VQ-GAN + Transformer pipeline.

</details>